# CRGCN trên REES46 Full (Temporal)

Notebook chạy CRGCN (https://github.com/MingshiYan/CRGCN) trên dataset REES46 Full (`rees46-full-temporal`) với:
- Mã CRGCN **clone trực tiếp từ GitHub** (`MingshiYan/CRGCN`) + cài `torch_geometric/torch_scatter/torch_sparse`.
- Hyperparams lấy **nguyên giá trị** từ `config/training.yaml` (embed_dim 256, n_layers 3, batch 8192, lr 8e-4, NDCG@10).
- Evaluation theo `evaluator.py` (full-rank, mask exclusion bằng `train_mask_purchase_only.pkl`, HR@k & NDCG@k) + phân khúc **cold/warm user** + **random fallback** cho cold (bốc ngẫu nhiên trong top-`COLD_POOL` item phổ biến → random nhưng không = 0).
- Lưu checkpoint lên Weights & Biases (fault-tolerant — key rỗng vẫn chạy bình thường).

## 1. Cài đặt môi trường + clone CRGCN

In [1]:
!pip install -q loguru tensorboard pyyaml
!pip install -q torch_geometric

import torch
TORCH = torch.__version__.split('+')[0]
CUDA = ('cu' + torch.version.cuda.replace('.', '')) if torch.version.cuda else 'cpu'
WHL = f'https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html'
print('Installing torch_scatter/torch_sparse from:', WHL)
!pip install -q torch_scatter torch_sparse -f {WHL}

import torch_scatter, torch_sparse
print('torch:', torch.__version__, '| torch_scatter:', torch_scatter.__version__, '| torch_sparse:', torch_sparse.__version__)

import os, sys
if not os.path.isdir('/content/CRGCN'):
    !git clone --depth 1 https://github.com/MingshiYan/CRGCN.git /content/CRGCN
sys.path.insert(0, '/content/CRGCN')
print('CRGCN files:', os.listdir('/content/CRGCN'))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.7 MB/s eta 0:00:00
Installing torch_scatter/torch_sparse from: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 40.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 108.2 MB/s eta 0:00:00
torch: 2.11.0+cu128 | torch_scatter: 2.1.2+pt211cu128 | torch_sparse: 0.6.18+pt211cu128
Cloning into '/content/CRGCN'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 14 (delta 0), reused 9 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 54.28 MiB | 15.35 MiB/s, done.
CRGCN files: ['.gitignore', 'requirements.txt', 'data.zip', 'data_set.py', 'gcn_conv.py', 'utils.py', 'trainer.py', 'mode

## 2. Tải dataset REES46 Full từ Hugging Face

In [2]:
import os
if not os.path.isdir('/content/data') or not os.path.isfile('/content/data/purchase_train_src.npy'):
    !pip install -q huggingface_hub hf_transfer
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id='nguyenmaiductrong/rees46-full-temporal',
        repo_type='dataset',
        local_dir='/content/data',
    )
print('data:', sorted(os.listdir('/content/data'))[:20])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 109.2 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:294: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 47 files:   0%|          | 0/47 [00:00<?, ?it/s]

data: ['.cache', '.gitattributes', 'SUBSAMPLE_MANIFEST.json', 'candidate_item_idx.npy', 'cart_train_dst.npy', 'cart_train_src.npy', 'cart_train_ts.npy', 'cart_trainval_dst.npy', 'cart_trainval_src.npy', 'cart_trainval_ts.npy', 'graph', 'item_is_cold.npy', 'node_counts.json', 'node_mappings', 'purchase_train_dst.npy', 'purchase_train_src.npy', 'purchase_train_ts.npy', 'purchase_trainval_dst.npy', 'purchase_trainval_src.npy', 'purchase_trainval_ts.npy']


## 3. Load `training.yaml`

In [3]:
import yaml, io, os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# YAML_TEXT = bản sao NGUYÊN VĂN của config/training.yaml.
# Notebook CRGCN chỉ đọc một phần field (data/model/training/evaluation/wandb);
# các section khác (cold_start/sampler/loss/hierarchy_cl/a100/...) giữ nguyên
# để khớp 100% training.yaml nhưng không ảnh hưởng tới pipeline CRGCN.
YAML_TEXT = """
data:
  data_dir: /content/data
  node_counts:
    brand: 3470
    category: 14
    product: 100775
    user: 245778
  struct_dir: /content/data/node_mappings

cold_start:
  content_item: true        # cong category+brand (da chuan hoa) vao input embedding product
  content_scale_init: 0.05  # scale hoc duoc, nho de content khong at id embedding (id norm ~0.07)
  p_id: 0.15                # xac suat drop ID embedding product khi train (mo phong cold item)
  p_hist: 0.2               # xac suat drop bot history user khi train (mo phong cold user)

model:
  dropout: 0.2
  embed_dim: 256
  n_layers: 3
  rank: 64
  use_grad_checkpoint: false
  n_intents: 64

sampler:
  hop1_budget: 16
  hop2_budget: 8
  hop1_sample_replace: true

loss:
  lambda_cl: 0.15
  lambda_conv: 0.1
  lambda_mono: 0.05
  funnel_margin: 0.1
  alpha: 0.5
  w_min: 0.05

hierarchy_cl:
  enabled: true
  tau: 0.1
  hard_k: 64
  min_pair_overlap: 4
  pair_weights: null

training:
  amp: true
  use_bf16: true
  batch_size: 8192
  device: cuda
  epochs: 40
  eval_batch_size: 8192
  eval_every: 1
  eval_subsample: 20000
  eval_seed: 42
  seed: 42
  deterministic: false
  l2_lambda: 1.0e-05
  lr: 1.0e-05
  min_lr: 1.0e-06
  warmup_epochs: 3
  max_grad_norm: 1.0
  cl_every_k: 2
  max_view_triplets: -1
  num_neg: 32
  num_workers: 8
  patience: 5
  save_dir: graph-recsys-v3
  weight_decay: 1.0e-02
  pin_memory: true
  persistent_workers: true
  prefetch_factor: 4

evaluation:
  full_ranking: true
  primary_metric: "NDCG@10"
  ks: [1, 5, 10, 20, 50]
  metrics: ["HR@1", "HR@5", "HR@10", "HR@20", "HR@50", "NDCG@1", "NDCG@5", "NDCG@10", "NDCG@20", "NDCG@50"]

wandb:
  artifact_name: graph-recsys-v3
  enabled: true
  entity: nguyenmaiductrong37-h-c-vi-n-c-ng-ngh-b-u-ch-nh-vi-n-th-ng
  project: graph-recsys
  run_name: graph-recsys-v3
  save_every: 1

a100:
  allow_tf32: true
  cudnn_benchmark: true
  use_fused_adamw: true
  compile_model: false
  empty_cache_freq: 0
"""

yaml_path = '/content/data/training.yaml'
if os.path.isfile(yaml_path):
    with open(yaml_path) as f:
        CFG = yaml.safe_load(f)
else:
    CFG = yaml.safe_load(io.StringIO(YAML_TEXT))

os.makedirs(CFG['training']['save_dir'], exist_ok=True)
print(yaml.safe_dump(CFG, sort_keys=False))

data:
  data_dir: /content/data
  node_counts:
    brand: 3470
    category: 14
    product: 100775
    user: 245778
  struct_dir: /content/data/node_mappings
cold_start:
  content_item: true
  content_scale_init: 0.05
  p_id: 0.15
  p_hist: 0.2
model:
  dropout: 0.2
  embed_dim: 256
  n_layers: 3
  rank: 64
  use_grad_checkpoint: false
  n_intents: 64
sampler:
  hop1_budget: 16
  hop2_budget: 8
  hop1_sample_replace: true
loss:
  lambda_cl: 0.15
  lambda_conv: 0.1
  lambda_mono: 0.05
  funnel_margin: 0.1
  alpha: 0.5
  w_min: 0.05
hierarchy_cl:
  enabled: true
  tau: 0.1
  hard_k: 64
  min_pair_overlap: 4
  pair_weights: null
training:
  amp: true
  use_bf16: true
  batch_size: 8192
  device: cuda
  epochs: 40
  eval_batch_size: 8192
  eval_every: 1
  eval_subsample: 20000
  eval_seed: 42
  seed: 42
  deterministic: false
  l2_lambda: 1.0e-05
  lr: 1.0e-05
  min_lr: 1.0e-06
  warmup_epochs: 3
  max_grad_norm: 1.0
  cl_every_k: 2
  max_view_triplets: -1
  num_neg: 32
  num_workers: 8
 

## 3b. Khởi tạo Weights & Biases

In [4]:
!pip install -q wandb
import os, wandb

# === W&B (tuỳ chọn) ===
# Dán API key của bạn vào WANDB_KEY để log online; để RỖNG -> chạy không log,
# KHÔNG chặn training. Key sai/hết hạn hoặc mất mạng cũng tự chuyển disabled.
WANDB_KEY = ''   # ví dụ: 'wandb_v1_xxx...'

WB_CFG = CFG.get('wandb', {}) or {}
ARTIFACT_NAME = WB_CFG.get('artifact_name', 'graph-recsys-v3')

def _init_wandb_online():
    if not WANDB_KEY:
        raise RuntimeError('WANDB_KEY rỗng')
    wandb.login(key=WANDB_KEY)
    return wandb.init(
        project=WB_CFG.get('project', 'graph-recsys'),
        entity=WB_CFG.get('entity', None),
        name=WB_CFG.get('run_name', 'graph-recsys-v3'),
        config=CFG,
    )

try:
    run = _init_wandb_online()
    print('W&B online:', run.url)
except Exception as e:
    print('[wandb] tắt logging (lý do:', repr(e)[:140], ')')
    os.environ['WANDB_MODE'] = 'disabled'
    run = wandb.init(mode='disabled', config=CFG)
    print('W&B: disabled — checkpoint vẫn lưu local ở', CFG['training']['save_dir'])

[wandb] tắt logging (lý do: RuntimeError('WANDB_KEY rỗng') )


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


W&B: disabled — checkpoint vẫn lưu local ở graph-recsys-v3


## 4. Load REES46 Full data → adapter cho CRGCN

CRGCN dùng index 1-based (padding_idx=0); REES46 dùng 0-based → shift `+1`.

In [5]:
import numpy as np, pickle, torch, json, os
from types import SimpleNamespace

DATA = CFG['data']['data_dir']
N_USERS = CFG['data']['node_counts']['user']
N_ITEMS = CFG['data']['node_counts']['product']

BEHAVIORS = ['view', 'cart', 'purchase']
# training.yaml chỉ định nghĩa max_view_triplets (-1 = dùng toàn bộ). cart/purchase
# không có trong config -> mặc định notebook (cart không cap thực tế, purchase dùng hết).
CAPS = {
    'view':     CFG['training'].get('max_view_triplets', -1),
    'cart':     CFG['training'].get('max_cart_triplets', -1),
    'purchase': CFG['training'].get('max_purchase_triplets', -1),
}
RNG = np.random.default_rng(CFG['training']['seed'])

def load_edges(name):
    src = np.load(os.path.join(DATA, f'{name}_train_src.npy')).astype(np.int64)
    dst = np.load(os.path.join(DATA, f'{name}_train_dst.npy')).astype(np.int64)
    cap = CAPS.get(name, -1)
    if cap and cap > 0 and len(src) > cap:   # cap <= 0 (vd -1) = dùng toàn bộ cạnh
        sel = RNG.choice(len(src), size=cap, replace=False)
        src, dst = src[sel], dst[sel]
        print(f'  {name}: subsampled to {cap:,} edges')
    return src, dst

edges = {b: load_edges(b) for b in BEHAVIORS}
for b, (s, d) in edges.items():
    print(f'{b}: edges={len(s):,}  u_max={s.max()}  i_max={d.max()}')

with open(os.path.join(DATA, 'train_mask_purchase_only.pkl'), 'rb') as f:
    TRAIN_MASK = pickle.load(f)
with open(os.path.join(DATA, 'val_ground_truth.pkl'), 'rb') as f:
    VAL_GT = pickle.load(f)
with open(os.path.join(DATA, 'test_ground_truth.pkl'), 'rb') as f:
    TEST_GT = pickle.load(f)
CAND = np.load(os.path.join(DATA, 'candidate_item_idx.npy')).astype(np.int64)
USER_COLD = np.load(os.path.join(DATA, 'user_is_cold.npy')).astype(bool)
# Độ phổ biến (theo purchase) -> dùng làm POOL cho cold-start random (random trong nhóm item dễ trúng)
ITEM_POP = np.bincount(edges['purchase'][1], minlength=N_ITEMS).astype(np.float32)
print(f'val users={len(VAL_GT):,}  test users={len(TEST_GT):,}  candidates={len(CAND):,}  '
      f'cold users={int(USER_COLD.sum()):,}/{len(USER_COLD):,}')

view: edges=3,500,126  u_max=184076  i_max=85665
cart: edges=970,492  u_max=184076  i_max=85665
purchase: edges=612,517  u_max=184094  i_max=85665
val users=15,392  test users=7,298  candidates=100,775  cold users=194,991/245,778


In [6]:
class BPATMPDataset:
    def __init__(self, n_users, n_items, edges_dict, behaviors):
        self.user_count = n_users
        self.item_count = n_items
        self.behaviors = behaviors
        self.edge_index = {}
        self.behavior_dict = {b: {} for b in behaviors}
        self.behavior_dict['all'] = {}
        self.user_behaviour_degree = []
        for b in behaviors:
            src, dst = edges_dict[b]
            u = torch.from_numpy(src + 1).long()
            i = torch.from_numpy(dst + 1).long()
            deg = torch.zeros(n_users + 1, dtype=torch.float32)
            deg.scatter_add_(0, u, torch.ones_like(u, dtype=torch.float32))
            self.user_behaviour_degree.append(deg.view(-1, 1))
            col = i + (n_users + 1)
            row_all = torch.cat([u, col])
            col_all = torch.cat([col, u])
            self.edge_index[b] = torch.stack([row_all, col_all])
            bd = {}
            for uu, ii in zip(src.tolist(), dst.tolist()):
                bd.setdefault(uu + 1, []).append(ii + 1)
            self.behavior_dict[b] = bd
        all_d = {}
        for b in behaviors:
            for u, lst in self.behavior_dict[b].items():
                all_d.setdefault(u, set()).update(lst)
        self.behavior_dict['all'] = {u: np.array(sorted(s), dtype=np.int64) for u, s in all_d.items()}
        self.user_behaviour_degree = torch.cat(self.user_behaviour_degree, dim=1)

dataset = BPATMPDataset(N_USERS, N_ITEMS, edges, BEHAVIORS)
print('Built dataset. edge_index keys:', list(dataset.edge_index.keys()))

Built dataset. edge_index keys: ['view', 'cart', 'purchase']


## 5. Khởi tạo model CRGCN

In [7]:
import random, numpy as np, torch
SEED = CFG['training']['seed']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

from model_cascade import CRGCN

device = CFG['training']['device'] if torch.cuda.is_available() else 'cpu'
args = SimpleNamespace(
    device=device,
    layers=[CFG['model']['n_layers']] * len(BEHAVIORS),
    node_dropout=CFG['model']['dropout'],
    message_dropout=CFG['model']['dropout'],
    embedding_size=CFG['model']['embed_dim'],
    behaviors=BEHAVIORS,
    reg_weight=CFG['training']['l2_lambda'],
    model_path=CFG['training']['save_dir'],
    check_point='',
    if_load_model=False,
)
model = CRGCN(args, dataset).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'model on {device} | params = {n_params/1e6:.2f}M')

model on cuda | params = 88.72M


## 6. Evaluator (port từ `evaluator.py`)
Full-rank, tiled, mask exclusion bằng `train_mask_purchase_only`, HR@k & NDCG@k.

In [8]:
@torch.no_grad()
def evaluate(model, gt_dict, exclude_dict, ks=(1,5,10,20,50), user_batch=512, item_tile=16384,
             user_cold=None, item_pop=None, cold_pool=500, rand_seed=42):
    model.eval()
    model.storage_all_embeddings = None
    max_k = max(ks)
    dev = next(model.parameters()).device
    dtype = torch.float16 if dev.type == 'cuda' else torch.float32

    model.storage_all_embeddings = model.gcn_propagate()
    last_b = model.behaviors[-1]
    u_all, i_all = torch.split(
        model.storage_all_embeddings[last_b],
        [model.n_users + 1, model.n_items + 1],
    )
    item_embs = i_all[1:N_ITEMS + 1].to(dtype)
    user_all = u_all.to(dtype)

    eval_uids = np.array(sorted(gt_dict.keys()), dtype=np.int64)
    n_eval = len(eval_uids)

    # Cold/warm user segmentation (True = cold), align theo thu tu eval_uids
    seg = None
    if user_cold is not None:
        seg = torch.as_tensor(np.asarray(user_cold)[eval_uids].astype(bool), device=dev)
        n_cold = int(seg.sum().item()); n_warm = n_eval - n_cold
        seg_sums = {f'{p}/{m}@{k}': 0.0 for p in ('warm_user', 'cold_user')
                    for m in ('HR', 'NDCG') for k in ks}

    # Cold-start fallback: cold user (lịch sử rỗng) được gợi ý NGẪU NHIÊN trong nhóm
    # top-`cold_pool` item phổ biến nhất -> random (mỗi user một list khác) nhưng KHÔNG = 0.
    # cold_pool nhỏ -> điểm cao hơn (gần popularity); lớn -> ngẫu nhiên hơn, điểm về 0.
    cold_gen = None; pop_pool = None
    if seg is not None and n_cold > 0 and item_pop is not None:
        cold_gen = torch.Generator(device=dev); cold_gen.manual_seed(int(rand_seed))
        pool_n = min(max(int(cold_pool), max_k), item_embs.size(0))
        pop_pool = torch.topk(torch.as_tensor(np.asarray(item_pop), device=dev, dtype=torch.float32), pool_n).indices

    gt_lists = [np.asarray(gt_dict[u], dtype=np.int64).reshape(-1) for u in eval_uids]
    max_pos = max(len(g) for g in gt_lists)
    gt_pad = torch.full((n_eval, max_pos), -1, dtype=torch.long, device=dev)
    gt_cnt = torch.zeros(n_eval, dtype=torch.long, device=dev)
    for r, g in enumerate(gt_lists):
        gt_cnt[r] = len(g)
        gt_pad[r, :len(g)] = torch.as_tensor(g, device=dev)

    uid2pos = {int(u): i for i, u in enumerate(eval_uids.tolist())}
    er, ec = [], []
    for u, items in exclude_dict.items():
        p = uid2pos.get(int(u))
        if p is None or len(items) == 0: continue
        items = np.asarray(items, dtype=np.int64).reshape(-1)
        er.extend([p] * len(items)); ec.extend(items.tolist())
    er = torch.as_tensor(er, dtype=torch.long, device=dev)
    ec = torch.as_tensor(ec, dtype=torch.long, device=dev)

    ndcg_w = 1.0 / torch.log2(torch.arange(1, max_k + 1, device=dev).float() + 1.0)
    sums = {f'{m}@{k}': 0.0 for m in ('HR', 'NDCG') for k in ks}

    for us in range(0, n_eval, user_batch):
        ue = min(us + user_batch, n_eval)
        B = ue - us
        uids_b = torch.as_tensor(eval_uids[us:ue] + 1, dtype=torch.long, device=dev)
        u_emb = user_all[uids_b]
        top_v = torch.full((B, max_k), float('-inf'), device=dev, dtype=dtype)
        top_i = torch.full((B, max_k), -1, device=dev, dtype=torch.long)
        inb = (er >= us) & (er < ue)
        sr, sc = er[inb] - us, ec[inb]
        n_items = item_embs.size(0)
        for ts in range(0, n_items, item_tile):
            te = min(ts + item_tile, n_items)
            tile = item_embs[ts:te]
            scores = u_emb @ tile.T
            tmask = (sc >= ts) & (sc < te)
            if tmask.any():
                scores[sr[tmask], sc[tmask] - ts] = float('-inf')
            k = min(max_k, scores.size(1))
            tv, ti = scores.topk(k, dim=-1)
            ti = ti + ts
            mv = torch.cat([top_v, tv], dim=-1)
            mi = torch.cat([top_i, ti], dim=-1)
            sel = mv.topk(max_k, dim=-1).indices
            top_v = mv.gather(1, sel); top_i = mi.gather(1, sel)
        seg_b = seg[us:ue] if seg is not None else None
        if pop_pool is not None and seg_b is not None and seg_b.any():
            c = int(seg_b.sum().item())
            # moi cold user: bốc max_k item PHAN BIET ngau nhien tu pop_pool
            pick = torch.rand(c, pop_pool.size(0), device=dev, generator=cold_gen).argsort(dim=1)[:, :max_k]
            top_i[seg_b] = pop_pool[pick]
        gtb = gt_pad[us:ue]; cnb = gt_cnt[us:ue]
        hits = (top_i.unsqueeze(-1) == gtb.unsqueeze(1)).any(dim=-1)
        for k in ks:
            hr_u = hits[:, :k].any(dim=-1).float()
            dcg = (hits[:, :k].float() * ndcg_w[:k]).sum(dim=-1)
            idl = torch.minimum(cnb, torch.full_like(cnb, k))
            idcg = torch.zeros_like(dcg)
            for v in idl.unique():
                mm = idl == v
                if int(v) > 0: idcg[mm] = ndcg_w[:int(v)].sum()
            ndcg_u = dcg / idcg.clamp_min(1e-12)
            sums[f'HR@{k}'] += hr_u.sum().item()
            sums[f'NDCG@{k}'] += ndcg_u.sum().item()
            if seg_b is not None:
                warm_m = ~seg_b
                seg_sums[f'cold_user/HR@{k}'] += hr_u[seg_b].sum().item()
                seg_sums[f'cold_user/NDCG@{k}'] += ndcg_u[seg_b].sum().item()
                seg_sums[f'warm_user/HR@{k}'] += hr_u[warm_m].sum().item()
                seg_sums[f'warm_user/NDCG@{k}'] += ndcg_u[warm_m].sum().item()
    model.storage_all_embeddings = None

    out = {k: v / n_eval for k, v in sums.items()}
    if seg is not None:
        for k in ks:
            if n_cold > 0:
                out[f'cold_user/HR@{k}'] = seg_sums[f'cold_user/HR@{k}'] / n_cold
                out[f'cold_user/NDCG@{k}'] = seg_sums[f'cold_user/NDCG@{k}'] / n_cold
            if n_warm > 0:
                out[f'warm_user/HR@{k}'] = seg_sums[f'warm_user/HR@{k}'] / n_warm
                out[f'warm_user/NDCG@{k}'] = seg_sums[f'warm_user/NDCG@{k}'] / n_warm
        out['cold_user/n'] = float(n_cold); out['warm_user/n'] = float(n_warm)
    return out

print('evaluate() defined (+ cold/warm segmentation + random-trong-pool-phổ-biến cho cold)')

evaluate() defined (+ cold/warm segmentation + random-trong-pool-phổ-biến cho cold)


## 7. Training loop

In [9]:
from torch.utils.data import Dataset, DataLoader
import time, gc

class BehaviorTripletDataset(Dataset):
    def __init__(self, dataset, behaviors, n_items):
        self.bd = dataset.behavior_dict
        self.behaviors = behaviors
        self.n_users = dataset.user_count
        self.n_items = n_items
    def __len__(self): return self.n_users
    def __getitem__(self, idx):
        u = idx + 1
        out = np.zeros((len(self.behaviors), 3), dtype=np.int64)
        all_pos = self.bd['all'].get(u)
        for k, b in enumerate(self.behaviors):
            lst = self.bd[b].get(u)
            if not lst: continue
            pos = lst[np.random.randint(len(lst))]
            neg = np.random.randint(1, self.n_items + 1)
            if all_pos is not None and len(all_pos) > 0:
                for _ in range(5):
                    if not np.isin(neg, all_pos): break
                    neg = np.random.randint(1, self.n_items + 1)
            out[k] = [u, pos, neg]
        return out

train_ds = BehaviorTripletDataset(dataset, BEHAVIORS, N_ITEMS)
train_loader = DataLoader(train_ds, batch_size=CFG['training']['batch_size'],
                          shuffle=True, num_workers=2, drop_last=True)

optim = torch.optim.Adam(model.parameters(), lr=CFG['training']['lr'])

best_metric = -1.0; best_epoch = -1
patience = CFG['training']['patience']; stale = 0
ks = CFG['evaluation']['ks']; primary = CFG['evaluation']['primary_metric']
save_dir = CFG['training']['save_dir']
best_ckpt_path = os.path.join(save_dir, 'best.pt')
COLD_POOL = 500   # random cold-start: bốc ngẫu nhiên trong top-COLD_POOL item phổ biến (nhỏ -> điểm cao hơn)

def upload_ckpt_to_wandb(path, epoch, metric_value, aliases=None):
    art = wandb.Artifact(ARTIFACT_NAME, type='model',
                         metadata={'epoch': epoch, primary: metric_value})
    art.add_file(path)
    run.log_artifact(art, aliases=aliases or ['latest'])

def free_vram():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

for epoch in range(1, CFG['training']['epochs'] + 1):
    model.train(); t0 = time.time(); ep_loss = 0.0; nb = 0
    for batch in train_loader:
        batch = batch.to(device)
        loss = model(batch)
        optim.zero_grad(); loss.backward(); optim.step()
        ep_loss += float(loss.item()); nb += 1
        del batch, loss
    free_vram()
    avg_loss = ep_loss / max(nb, 1); dt = time.time() - t0
    print(f'epoch {epoch:03d} | loss {avg_loss:.4f} | {dt:.1f}s')
    log_payload = {'epoch': epoch, 'train/loss': avg_loss, 'train/epoch_seconds': dt}

    if epoch % CFG['training']['eval_every'] == 0:
        m = evaluate(model, VAL_GT, TRAIN_MASK, ks=ks,
                     user_batch=CFG['training']['eval_batch_size'], user_cold=USER_COLD,
                     item_pop=ITEM_POP, cold_pool=COLD_POOL)
        free_vram()
        print('  val:', {k: round(v, 4) for k, v in m.items() if '/' not in k})
        if f'cold_user/{primary}' in m:
            print(f"        cold(n={int(m.get('cold_user/n',0))}) {primary}="
                  f"{m.get(f'cold_user/{primary}',float('nan')):.4f} | "
                  f"warm(n={int(m.get('warm_user/n',0))}) {primary}="
                  f"{m.get(f'warm_user/{primary}',float('nan')):.4f}")
        for k, v in m.items(): log_payload[f'val/{k}'] = v
        cur = m[primary]
        if cur > best_metric:
            best_metric = cur; best_epoch = epoch; stale = 0
            torch.save(model.state_dict(), best_ckpt_path)
            upload_ckpt_to_wandb(best_ckpt_path, epoch, cur, aliases=['best', 'latest'])
            print(f'new best {primary}={cur:.4f} (saved + uploaded)')
            log_payload['val/best_metric'] = cur
            log_payload['val/best_epoch'] = epoch
        else:
            stale += 1
            wandb.log(log_payload, step=epoch)
            if stale >= patience:
                print(f'  early stop at epoch {epoch} (best epoch {best_epoch})')
                break
            continue
    wandb.log(log_payload, step=epoch)

print(f'\nDone. best {primary}={best_metric:.4f} @ epoch {best_epoch}')

epoch 001 | loss 1.7675 | 14.3s
  val: {'HR@1': 0.0248, 'HR@5': 0.0777, 'HR@10': 0.1218, 'HR@20': 0.1791, 'HR@50': 0.2855, 'NDCG@1': 0.0248, 'NDCG@5': 0.033, 'NDCG@10': 0.0414, 'NDCG@20': 0.0519, 'NDCG@50': 0.0705}
        cold(n=3022) NDCG@10=0.0047 | warm(n=12370) NDCG@10=0.0504
new best NDCG@10=0.0414 (saved + uploaded)
epoch 002 | loss 1.7352 | 13.8s
  val: {'HR@1': 0.027, 'HR@5': 0.0871, 'HR@10': 0.1305, 'HR@20': 0.1976, 'HR@50': 0.2972, 'NDCG@1': 0.027, 'NDCG@5': 0.0373, 'NDCG@10': 0.046, 'NDCG@20': 0.0594, 'NDCG@50': 0.0779}
        cold(n=3022) NDCG@10=0.0047 | warm(n=12370) NDCG@10=0.0561
new best NDCG@10=0.0460 (saved + uploaded)
epoch 003 | loss 1.7270 | 13.8s
  val: {'HR@1': 0.0288, 'HR@5': 0.094, 'HR@10': 0.14, 'HR@20': 0.2065, 'HR@50': 0.3031, 'NDCG@1': 0.0288, 'NDCG@5': 0.0401, 'NDCG@10': 0.0495, 'NDCG@20': 0.0632, 'NDCG@50': 0.0813}
        cold(n=3022) NDCG@10=0.0047 | warm(n=12370) NDCG@10=0.0604
new best NDCG@10=0.0495 (saved + uploaded)
epoch 004 | loss 1.7243 | 13.

## 8. Đánh giá trên test set với checkpoint tốt nhất

In [10]:
ckpt = os.path.join(save_dir, 'best.pt')
if os.path.isfile(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=device))
    print('loaded', ckpt)
COLD_POOL = globals().get('COLD_POOL', 500)   # đồng bộ với cell train; fallback 500 nếu chạy riêng
m_test = evaluate(model, TEST_GT, TRAIN_MASK, ks=CFG['evaluation']['ks'],
                  user_batch=CFG['training']['eval_batch_size'], user_cold=USER_COLD,
                  item_pop=ITEM_POP, cold_pool=COLD_POOL)
print('TEST metrics:')
for k in CFG['evaluation']['ks']:
    print(f'  HR@{k:>3} = {m_test[f"HR@{k}"]:.4f}   NDCG@{k:>3} = {m_test[f"NDCG@{k}"]:.4f}')

for seg_name in ('cold_user', 'warm_user'):
    if f'{seg_name}/n' not in m_test: continue
    print(f'\n{seg_name} (n={int(m_test[f"{seg_name}/n"])}):')
    for k in CFG['evaluation']['ks']:
        hr = m_test.get(f'{seg_name}/HR@{k}', float('nan'))
        nd = m_test.get(f'{seg_name}/NDCG@{k}', float('nan'))
        print(f'  HR@{k:>3} = {hr:.4f}   NDCG@{k:>3} = {nd:.4f}')

wandb.log({f'test/{k}': v for k, v in m_test.items()})
wandb.summary.update({f'test/{k}': v for k, v in m_test.items()})
wandb.summary['best_val_epoch'] = best_epoch
wandb.summary[f'best_val_{primary}'] = best_metric

final_art = wandb.Artifact(ARTIFACT_NAME, type='model',
                           metadata={'best_epoch': best_epoch,
                                     f'val_{primary}': best_metric,
                                     **{f'test_{k}': v for k, v in m_test.items()}})
final_art.add_file(ckpt)
run.log_artifact(final_art, aliases=['final'])
wandb.finish()

loaded graph-recsys-v3/best.pt
TEST metrics:
  HR@  1 = 0.0104   NDCG@  1 = 0.0104
  HR@  5 = 0.0459   NDCG@  5 = 0.0204
  HR@ 10 = 0.0713   NDCG@ 10 = 0.0264
  HR@ 20 = 0.1189   NDCG@ 20 = 0.0358
  HR@ 50 = 0.1973   NDCG@ 50 = 0.0496

cold_user (n=1446):
  HR@  1 = 0.0021   NDCG@  1 = 0.0021
  HR@  5 = 0.0069   NDCG@  5 = 0.0032
  HR@ 10 = 0.0118   NDCG@ 10 = 0.0043
  HR@ 20 = 0.0207   NDCG@ 20 = 0.0063
  HR@ 50 = 0.0615   NDCG@ 50 = 0.0127

warm_user (n=5852):
  HR@  1 = 0.0125   NDCG@  1 = 0.0125
  HR@  5 = 0.0555   NDCG@  5 = 0.0247
  HR@ 10 = 0.0860   NDCG@ 10 = 0.0319
  HR@ 20 = 0.1432   NDCG@ 20 = 0.0431
  HR@ 50 = 0.2309   NDCG@ 50 = 0.0588
